# Orgãos e Titulos

In [ ]:
import requests
from collections import Counter

def get_deputados():
    deps = []
    pagina = 1
    while True:
        r = requests.get(
            "https://dadosabertos.camara.leg.br/api/v2/deputados",
            params={"pagina": pagina, "itens": 100},
            headers={"accept": "application/json"}
        ).json()
        dados = r.get("dados", [])
        if not dados:
            break
        deps.extend(dados)
        pagina += 1
    return deps

titulos_encontrados = Counter()

deputados = get_deputados()
for dep in deputados[:50]:  # começa com 50 pra ser rápido
    r = requests.get(
        f"https://dadosabertos.camara.leg.br/api/v2/deputados/{dep['id']}/orgaos",
        headers={"accept": "application/json"}
    ).json()
    for item in r.get("dados", []):
        titulo    = item.get("titulo", "")
        cod       = item.get("codTitulo", "")
        titulos_encontrados[f"{cod} | {titulo}"] += 1

for titulo, qtd in titulos_encontrados.most_common():
    print(f"{qtd:>4}x  {titulo}")

 176x  101 | Titular
  96x  102 | Suplente
   8x  1 | Presidente
   7x  2 | 1º Vice-Presidente
   5x  50 | Relator
   3x  4 | 3º Vice-Presidente
   3x  14 | Coordenador
   2x  3 | 2º Vice-Presidente
   1x  30 | 3º Coordenador Adjunto
   1x  9 | 1º Suplente de Secretário
   1x  18 | Subcoordenador


Criei uma Unique key para evitar que duplicatas e vai inserir a mesma linha várias vezes se você rodar o script mais de uma vez. 

In [ ]:
cursor = conn.cursor()

try:
    cursor.execute("""
        ALTER TABLE lideranca_orgaos
        ADD UNIQUE KEY uq_dep_orgao_inicio (fk_deputado, cd_orgao, data_inicio)
    """)
    conn.commit()
    print("✅ UNIQUE KEY criada com sucesso!")
except mysql.connector.errors.DatabaseError as e:
    if "Duplicate key name" in str(e):
        print("⚠ UNIQUE KEY já existe, nada foi alterado.")
    else:
        print(f"❌ Erro: {e}")
finally:
    cursor.close()
    conn.close()

✅ UNIQUE KEY criada com sucesso!


Alimentando a tabela lideranca_orgaos

In [ ]:
cursor = conn.cursor()

SIGLA_CARGO = {
    "1":   "PRES",
    "2":   "VICE1",
    "3":   "VICE2",
    "4":   "VICE3",
    "14":  "COORD",
    "50":  "REL",
    "101": "TIT",
    "102": "SUP",
    "30":  "COORD3",
    "9":   "SUP1SEC",
    "18":  "SUBCOORD",
}

def parse_date(valor):
    if not valor:
        return None
    try:
        return datetime.fromisoformat(valor).date()
    except ValueError:
        return None

def get_deputados():
    deps = []
    pagina = 1
    while True:
        r = requests.get(
            "https://dadosabertos.camara.leg.br/api/v2/deputados",
            params={"pagina": pagina, "itens": 100, "ordem": "ASC", "ordenarPor": "nome"},
            headers={"accept": "application/json"}
        ).json()
        dados = r.get("dados", [])
        if not dados:
            break
        deps.extend(dados)
        pagina += 1
    return deps

def get_orgaos_deputado(id_deputado):
    r = requests.get(
        f"https://dadosabertos.camara.leg.br/api/v2/deputados/{id_deputado}/orgaos",
        params={"itens": 100, "ordem": "ASC", "ordenarPor": "dataInicio"},
        headers={"accept": "application/json"}
    ).json()
    return r.get("dados", [])


deputados = get_deputados()
total = len(deputados)
print(f"Total de deputados: {total}")

for i, dep in enumerate(deputados, 1):
    id_dep = dep["id"]
    print(f"[{i}/{total}] {dep['nome']}")

    try:
        orgaos = get_orgaos_deputado(id_dep)

        for item in orgaos:
            cod_cargo = item.get("codTitulo", "")
            sigla     = SIGLA_CARGO.get(cod_cargo, "?")

            cursor.execute("""
                INSERT INTO lideranca_orgaos
                    (fk_deputado, cd_orgao, sigla_orgao, nome_orgao,
                     cargo, cd_cargo, sigla_cargo,
                     data_inicio, data_fim, peso_cargo)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, NULL)
                ON DUPLICATE KEY UPDATE
                    cargo       = VALUES(cargo),
                    cd_cargo    = VALUES(cd_cargo),
                    sigla_cargo = VALUES(sigla_cargo),
                    data_fim    = VALUES(data_fim),
                    peso_cargo  = peso_cargo
            """, (
                id_dep,
                item["idOrgao"],
                item.get("siglaOrgao"),
                item.get("nomeOrgao"),
                item.get("titulo"),
                cod_cargo,
                sigla,
                parse_date(item.get("dataInicio")),
                parse_date(item.get("dataFim"))
            ))

        conn.commit()
        time.sleep(0.2)

    except Exception as e:
        print(f"  ⚠ Erro no deputado {id_dep}: {e}")
        conn.rollback()

print("✅ Concluído!")
cursor.close()
conn.close()

Buscando o Tipo_Orgao

In [ ]:
cursor = conn.cursor()

# Busca apenas os cd_orgao únicos que estão sem tipo
cursor.execute("SELECT DISTINCT cd_orgao FROM lideranca_orgaos WHERE tipo_orgao IS NULL")
orgaos = cursor.fetchall()
total = len(orgaos)
print(f"Total de órgãos para atualizar: {total}")

for i, (cd_orgao,) in enumerate(orgaos, 1):
    try:
        r = requests.get(
            f"https://dadosabertos.camara.leg.br/api/v2/orgaos/{cd_orgao}",
            headers={"accept": "application/json"}
        ).json()

        tipo = r.get("dados", {}).get("tipoOrgao")

        if tipo:
            cursor.execute("""
                UPDATE lideranca_orgaos
                SET tipo_orgao = %s
                WHERE cd_orgao = %s
            """, (tipo, cd_orgao))
            conn.commit()
            print(f"[{i}/{total}] {cd_orgao} → {tipo}")
        else:
            print(f"[{i}/{total}] {cd_orgao} → sem tipo_orgao na API")

        time.sleep(0.15)

    except Exception as e:
        print(f"  ⚠ Erro no órgão {cd_orgao}: {e}")

print("✅ Concluído!")
cursor.close()
conn.close()